# Domain F: Recurrent Neural Network Modeling

This notebook addresses two questions:
- **F1:** Can a task-trained RNN (Dale's law constrained, E/I ratio matching data) reproduce cell-type-specific tuning and connectivity?
- **F2:** Can a data-driven RNN trained to predict population ΔF/F learn biologically meaningful representations?

**Dependencies:** `torch` (PyTorch). Install via `pip install torch`.

**Note:** Both models use the actual stimulus parameters from the dataset but are trained on synthetic or simplified tasks. Sections requiring additional data or long training are marked.

In [ ]:
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
    print(f"PyTorch version: {torch.__version__}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️ PyTorch not installed. Install via: pip install torch")

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

def load_mouse_zarr(mouse_id):
    """Load one mouse's data from zarr, returning an adata-like SimpleNamespace."""
    z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
    ct = z['transcriptomics/cell_type']
    morph = z['morphology/mask_properties']
    obs_dict = {
        'unique_id': z['unique_id'][:].astype(str),
        'mouse_id': mouse_id,
        'class_name': ct['class_name'][:],
        'subclass_name': ct['subclass_name'][:],
        'subclass_label': ct['subclass_label'][:],
        'supertype_name': ct['supertype_name'][:],
        'cluster_name': ct['cluster_name'][:],
        'cluster_alias': ct['cluster_alias'][:],
        'x_loc': morph['centroid_x_um'][:],
        'y_loc': morph['centroid_y_um'][:],
        'z_loc': morph['centroid_z_um'][:],
    }
    obs_df = pd.DataFrame(obs_dict)
    n_cells = len(obs_df)

    X_sessions, var_sessions = [], []
    for si, sess in enumerate(SESSIONS):
        gs = z[f'ophys/drifting_gratings/{sess}/stim_aligned_dff/GratingStim']
        dff = gs['dff'][:]
        time_rel = gs['time_relative'][:]
        running = gs['running'][:]
        gray = gs['trial_info/gray_screen'][:]
        valid = ~gray
        stim_mask = (time_rel >= 0) & (time_rel <= 2.0)
        dff_avg = dff[valid][:, stim_mask, :].mean(axis=1)
        run_speed = running[valid][:, stim_mask, 0].mean(axis=1)
        X_sessions.append(dff_avg.T)
        var_sessions.append(pd.DataFrame({
            'contrast': gs['trial_info/contrast'][:][valid],
            'orientation': gs['trial_info/orientation'][:][valid],
            'temporal_frequency': gs['trial_info/temporal_frequency'][:][valid],
            'spatial_frequency': gs['trial_info/spatial_frequency'][:][valid],
            'stim_block': gs['trial_info/stim_block'][:][valid],
            'stim_name': gs['trial_info/stim_name'][:][valid],
            'start_time': gs['trial_info/start_time'][:][valid],
            'stop_time': gs['trial_info/stop_time'][:][valid],
            'duration': gs['trial_info/duration'][:][valid],
            'avg_running': run_speed,
            'is_running': run_speed > 1.0,
            'day': si + 1,
        }))
    return SimpleNamespace(
        X=np.hstack(X_sessions), obs=obs_df, var=pd.concat(var_sessions, ignore_index=True),
        n_obs=n_cells, n_vars=sum(v.shape[0] for v in var_sessions)
    )

adata_list = [load_mouse_zarr(mid) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])

# ── Data-derived E/I ratio ──
sc_counts = obs['subclass_name'].value_counts()
exc_count = sum(sc_counts.get(s, 0) for s in present_subclasses if 'Glut' in s)
inh_count = sum(sc_counts.get(s, 0) for s in present_subclasses if 'Gaba' in s)
print(f"\nExcitatory: {exc_count} ({exc_count/len(obs):.1%}), Inhibitory: {inh_count} ({inh_count/len(obs):.1%})")
print(f"E/I ratio: {exc_count/max(inh_count,1):.1f}:1")

## F1: Task-Trained RNN with Dale's Law

Build an RNN with separate excitatory and inhibitory units (Dale's law), matching the observed E/I ratio. Train on orientation classification. Then analyze the trained network's tuning curves, noise correlations, and response to unit-type ablation.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F1.1  Define Dale's-Law RNN architecture
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    class DalesRNN(nn.Module):
        """Continuous-time RNN with Dale's law constraint.
        
        Excitatory units have non-negative outgoing weights.
        Inhibitory units have non-positive outgoing weights.
        
        Architecture:
          - Input: (orientation, contrast, TF, running_speed) → 4 dims
          - Recurrent: N_E excitatory + N_I inhibitory units
          - Output: 8-way orientation classification
        """
        
        def __init__(self, n_exc=200, n_pvalb=20, n_sst=10, n_vip=10, n_lamp5=5,
                     n_input=4, n_output=8, dt=0.1, tau=1.0, noise_std=0.05):
            super().__init__()
            
            self.n_exc = n_exc
            self.n_inh = n_pvalb + n_sst + n_vip + n_lamp5
            self.n_total = n_exc + self.n_inh
            self.n_input = n_input
            self.n_output = n_output
            self.dt = dt
            self.tau = tau
            self.noise_std = noise_std
            
            # Track unit types
            self.unit_types = (
                ['L2/3 IT'] * (n_exc // 2) +
                ['L4/5 IT'] * (n_exc - n_exc // 2) +
                ['Pvalb'] * n_pvalb +
                ['Sst'] * n_sst +
                ['Vip'] * n_vip +
                ['Lamp5'] * n_lamp5
            )
            
            # Dale's law mask: +1 for exc, -1 for inh
            dale_mask = torch.ones(self.n_total)
            dale_mask[n_exc:] = -1
            self.register_buffer('dale_mask', dale_mask)
            
            # Learnable weight matrices (stored as non-negative, sign applied via dale_mask)
            self.W_rec_raw = nn.Parameter(torch.abs(torch.randn(self.n_total, self.n_total) * 0.1))
            self.W_in = nn.Parameter(torch.randn(self.n_total, n_input) * 0.3)
            self.W_out = nn.Parameter(torch.randn(n_output, self.n_total) * 0.3)
            
            self.bias = nn.Parameter(torch.zeros(self.n_total))
            self.b_out = nn.Parameter(torch.zeros(n_output))
        
        def get_effective_W(self):
            """Apply Dale's law: W_eff[i, j] = |W_raw[i, j]| * dale_mask[j]"""
            return torch.abs(self.W_rec_raw) * self.dale_mask.unsqueeze(0)
        
        def forward(self, inputs, n_steps=20):
            """
            inputs: (batch, n_input) — stimulus parameters for one trial
            Returns: outputs (batch, n_steps, n_output), rates (batch, n_steps, n_total)
            """
            batch_size = inputs.shape[0]
            h = torch.zeros(batch_size, self.n_total, device=inputs.device)
            W_eff = self.get_effective_W()
            
            all_rates = []
            all_outputs = []
            
            for t in range(n_steps):
                # Noise injection
                noise = self.noise_std * torch.randn_like(h) if self.training else 0
                
                # RNN dynamics: tau * dh/dt = -h + W_eff @ r + W_in @ u + b
                r = torch.relu(h)  # firing rate (rectified)
                dh = (-h + r @ W_eff.T + inputs @ self.W_in.T + self.bias + noise) * (self.dt / self.tau)
                h = h + dh
                
                all_rates.append(r)
                output = h @ self.W_out.T + self.b_out  # linear readout
                all_outputs.append(output)
            
            rates = torch.stack(all_rates, dim=1)  # (batch, n_steps, n_total)
            outputs = torch.stack(all_outputs, dim=1)  # (batch, n_steps, n_output)
            return outputs, rates
    
    print("DalesRNN model defined.")
    print(f"Architecture: {200} exc + {20+10+10+5} inh = {245} units → 8 orientations")
else:
    print("⚠️ Skipping model definition — PyTorch not available.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F1.2  Prepare stimulus inputs and train the RNN
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    # Build stimulus dataset from actual experimental parameters
    stim_ori = var['orientation'].values.astype(float)
    stim_contrast = var['contrast'].values.astype(float)
    stim_tf = var['temporal_frequency'].values.astype(float)
    stim_running = var['avg_running'].values.astype(float)
    
    # Encode orientation as sin/cos (circular), normalize others
    ori_rad = np.deg2rad(stim_ori)
    input_features = np.column_stack([
        np.cos(ori_rad),            # orientation component 1
        np.sin(ori_rad),            # orientation component 2  
        stim_contrast / 0.8,        # normalized contrast
        np.log2(stim_tf) / 4,       # log-normalized TF
    ])
    
    # Labels: 8-way orientation classification
    ori_to_label = {o: i for i, o in enumerate(orientations)}
    labels = np.array([ori_to_label[o] for o in stim_ori])
    
    # Convert to tensors
    X_stim = torch.FloatTensor(input_features).to(device)
    y_labels = torch.LongTensor(labels).to(device)
    
    # Train/test split
    n_trials = len(labels)
    np.random.seed(42)
    perm = np.random.permutation(n_trials)
    train_idx = perm[:int(0.8 * n_trials)]
    test_idx = perm[int(0.8 * n_trials):]
    
    # Initialize model
    model = DalesRNN(n_exc=200, n_pvalb=20, n_sst=10, n_vip=10, n_lamp5=5).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    
    # Training loop
    n_epochs = 100
    batch_size = 256
    losses = []
    
    print("Training task-trained RNN...")
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        np.random.shuffle(train_idx)
        
        for start in range(0, len(train_idx), batch_size):
            batch_idx = train_idx[start:start+batch_size]
            x_batch = X_stim[batch_idx]
            y_batch = y_labels[batch_idx]
            
            outputs, rates = model(x_batch, n_steps=20)
            # Use output at last time step
            logits = outputs[:, -1, :]
            
            loss = criterion(logits, y_batch)
            # Add L2 rate penalty for biological plausibility
            rate_penalty = 1e-4 * rates.pow(2).mean()
            total_loss = loss + rate_penalty
            
            optimizer.zero_grad()
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        losses.append(epoch_loss / max(1, len(train_idx) // batch_size))
        
        if (epoch + 1) % 20 == 0:
            # Test accuracy
            model.eval()
            with torch.no_grad():
                outputs_test, _ = model(X_stim[test_idx], n_steps=20)
                logits_test = outputs_test[:, -1, :]
                preds = logits_test.argmax(dim=1)
                acc = (preds == y_labels[test_idx]).float().mean().item()
            print(f"  Epoch {epoch+1}/{n_epochs}: loss={losses[-1]:.4f}, test_acc={acc:.3f}")
    
    # ── Training curve ──
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(losses, 'b-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('F1: Training Loss — Task-Trained RNN', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"Final test accuracy: {acc:.3f}")
else:
    print("⚠️ Skipping training — PyTorch not available.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F1.3  Analyze trained RNN: tuning curves, noise correlations, ablation
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    model.eval()
    
    # ── Extract unit tuning curves ──
    # Present each unique orientation (at contrast=0.8, TF=1) and measure firing rates
    unit_ori_tuning = np.zeros((model.n_total, len(orientations)))
    
    for oi, ori in enumerate(orientations):
        ori_rad = np.deg2rad(ori)
        stim_input = torch.FloatTensor([[np.cos(ori_rad), np.sin(ori_rad), 1.0, 0.0]]).to(device)
        # Run multiple times for noise averaging
        rates_samples = []
        for _ in range(50):
            model.train()  # enable noise
            with torch.no_grad():
                _, rates = model(stim_input, n_steps=20)
                rates_samples.append(rates[0, -1, :].cpu().numpy())
        model.eval()
        unit_ori_tuning[:, oi] = np.mean(rates_samples, axis=0)
    
    # ── Plot tuning curves by unit type ──
    unit_type_arr = np.array(model.unit_types)
    unique_types = ['L2/3 IT', 'L4/5 IT', 'Pvalb', 'Sst', 'Vip', 'Lamp5']
    type_colors = {'L2/3 IT': '#1f77b4', 'L4/5 IT': '#2ca02c', 'Pvalb': '#d62728',
                   'Sst': '#ff7f0e', 'Vip': '#e377c2', 'Lamp5': '#8c564b'}
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, utype in zip(axes.flat, unique_types):
        type_mask = unit_type_arr == utype
        n_units = type_mask.sum()
        if n_units == 0:
            ax.set_visible(False)
            continue
        
        tuning = unit_ori_tuning[type_mask]
        mean_t = np.mean(tuning, axis=0)
        sem_t = np.std(tuning, axis=0) / np.sqrt(n_units)
        
        ax.errorbar(orientations, mean_t, yerr=sem_t, color=type_colors[utype],
                    linewidth=2, capsize=3, marker='o')
        # Show individual units
        for i in range(min(20, n_units)):
            ax.plot(orientations, tuning[i], color=type_colors[utype], alpha=0.15, linewidth=0.5)
        
        ax.set_title(f'{utype} (n={n_units})', color=type_colors[utype], fontweight='bold')
        ax.set_xlabel('Direction (°)')
        ax.set_ylabel('Firing Rate')
        ax.set_xticks(orientations)
    
    plt.suptitle('F1: RNN Unit Tuning Curves by Type', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # ── Effective weight matrix visualization ──
    W_eff = model.get_effective_W().detach().cpu().numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    ax = axes[0]
    im = ax.imshow(W_eff, aspect='auto', cmap='RdBu_r', vmin=-np.percentile(np.abs(W_eff), 95),
                   vmax=np.percentile(np.abs(W_eff), 95))
    plt.colorbar(im, ax=ax)
    # Mark E/I boundary
    ax.axhline(model.n_exc - 0.5, color='yellow', lw=1)
    ax.axvline(model.n_exc - 0.5, color='yellow', lw=1)
    ax.set_title('F1: Effective Recurrent Weights', fontweight='bold')
    ax.set_xlabel('From unit')
    ax.set_ylabel('To unit')
    
    # E→E, E→I, I→E, I→I block means
    ax = axes[1]
    ne = model.n_exc
    block_means = {
        'E→E': np.mean(W_eff[:ne, :ne]),
        'E→I': np.mean(W_eff[ne:, :ne]),
        'I→E': np.mean(W_eff[:ne, ne:]),
        'I→I': np.mean(W_eff[ne:, ne:]),
    }
    ax.bar(block_means.keys(), block_means.values(),
           color=['green', 'orange', 'red', 'purple'])
    ax.set_ylabel('Mean Weight')
    ax.set_title('F1: Mean Connection Weights by Block', fontweight='bold')
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    
    plt.tight_layout()
    plt.show()
    
    # ── Ablation experiment ──
    print("\n=== Ablation Experiment ===")
    # Baseline accuracy
    with torch.no_grad():
        outputs_base, _ = model(X_stim[test_idx], n_steps=20)
        base_acc = (outputs_base[:, -1, :].argmax(1) == y_labels[test_idx]).float().mean().item()
    print(f"Baseline accuracy: {base_acc:.3f}")
    
    ablation_results = {}
    for abl_type in unique_types:
        # Zero out firing rates of this unit type
        type_mask_t = torch.tensor([t == abl_type for t in model.unit_types], dtype=torch.bool)
        
        # Temporarily modify forward pass by masking rates
        saved_bias = model.bias.data.clone()
        # Set bias to large negative for ablated units (effectively silencing them)
        model.bias.data[type_mask_t] = -100.0
        
        with torch.no_grad():
            outputs_abl, _ = model(X_stim[test_idx], n_steps=20)
            abl_acc = (outputs_abl[:, -1, :].argmax(1) == y_labels[test_idx]).float().mean().item()
        
        model.bias.data = saved_bias  # restore
        drop = base_acc - abl_acc
        ablation_results[abl_type] = {'acc': abl_acc, 'acc_drop': drop}
        print(f"  Ablate {abl_type:10s}: acc={abl_acc:.3f}, drop={drop:+.3f}")
    
    # ── Ablation bar chart ──
    fig, ax = plt.subplots(figsize=(10, 5))
    types_list = list(ablation_results.keys())
    drops = [ablation_results[t]['acc_drop'] for t in types_list]
    colors_abl = [type_colors.get(t, 'gray') for t in types_list]
    ax.bar(types_list, drops, color=colors_abl)
    ax.set_ylabel('Accuracy Drop')
    ax.set_title('F1: Ablation Impact on Orientation Decoding', fontweight='bold')
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Skipping analysis — PyTorch not available.")

## F2: Data-Driven RNN — Predict Population ΔF/F

Train an RNN to predict the actual population neural activity from stimulus inputs and running speed. Compare learned internal representations to real population coding geometry using CKA and RSA.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F2.1  Data-driven RNN: predict population ΔF/F from stimulus
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    class PredictiveRNN(nn.Module):
        """RNN that predicts population ΔF/F from stimulus + running input."""
        
        def __init__(self, n_input=5, n_hidden=256, n_output=100, n_layers=1):
            super().__init__()
            self.n_hidden = n_hidden
            self.rnn = nn.GRU(n_input, n_hidden, n_layers, batch_first=True)
            self.readout = nn.Linear(n_hidden, n_output)
        
        def forward(self, x, h0=None):
            """x: (batch, seq_len, n_input) → pred: (batch, seq_len, n_output)"""
            out, hn = self.rnn(x, h0)
            pred = self.readout(out)
            return pred, out  # pred = predicted ΔF/F, out = hidden states
    
    # ── Prepare data: use single mouse, subsample cells for tractability ──
    mouse_pick = obs['mouse_id'].unique()[np.argmax(
        [np.sum(obs['mouse_id'].values == m) for m in obs['mouse_id'].unique()])]
    m_mask = obs['mouse_id'].values == mouse_pick
    m_X = X[m_mask]
    n_cells_mouse = m_mask.sum()
    
    # Subsample to 100 cells if needed
    n_target = min(100, n_cells_mouse)
    np.random.seed(42)
    cell_idx = np.sort(np.random.choice(n_cells_mouse, n_target, replace=False))
    target_dff = m_X[cell_idx]  # (n_target, n_trials)
    target_subclasses = obs['subclass_name'].values[m_mask][cell_idx]
    
    # Stimulus input features per trial: cos(ori), sin(ori), contrast, log(TF), running
    ori_rad = np.deg2rad(var['orientation'].values.astype(float))
    running_speed = var['avg_running'].values.astype(float)
    running_speed = np.clip(running_speed / 30, -1, 2)  # normalize
    
    stim_inputs = np.column_stack([
        np.cos(ori_rad),
        np.sin(ori_rad),
        var['contrast'].values.astype(float) / 0.8,
        np.log2(var['temporal_frequency'].values.astype(float)) / 4,
        running_speed,
    ])
    
    # Treat each trial as a single time step (sequence length=1 per trial)
    # Or group consecutive trials into sequences
    seq_len = 20  # group 20 consecutive trials
    n_trials = stim_inputs.shape[0]
    n_seqs = n_trials // seq_len
    
    X_seqs = stim_inputs[:n_seqs * seq_len].reshape(n_seqs, seq_len, -1)
    Y_seqs = target_dff[:, :n_seqs * seq_len].T.reshape(n_seqs, seq_len, -1)
    
    X_tensor = torch.FloatTensor(X_seqs).to(device)
    Y_tensor = torch.FloatTensor(Y_seqs).to(device)
    
    # Train/test split
    n_train = int(0.8 * n_seqs)
    X_train, X_test = X_tensor[:n_train], X_tensor[n_train:]
    Y_train, Y_test = Y_tensor[:n_train], Y_tensor[n_train:]
    
    # Initialize and train
    pred_model = PredictiveRNN(n_input=5, n_hidden=256, n_output=n_target).to(device)
    optimizer2 = optim.Adam(pred_model.parameters(), lr=1e-3)
    
    print(f"Training data-driven RNN: {n_train} sequences of length {seq_len}")
    print(f"Predicting {n_target} cells from 5 stimulus features")
    
    losses2 = []
    for epoch in range(200):
        pred_model.train()
        pred, hidden = pred_model(X_train)
        loss = nn.MSELoss()(pred, Y_train)
        
        optimizer2.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(pred_model.parameters(), 1.0)
        optimizer2.step()
        
        losses2.append(loss.item())
        if (epoch + 1) % 50 == 0:
            pred_model.eval()
            with torch.no_grad():
                pred_test, _ = pred_model(X_test)
                test_loss = nn.MSELoss()(pred_test, Y_test).item()
            print(f"  Epoch {epoch+1}: train_loss={losses2[-1]:.4f}, test_loss={test_loss:.4f}")
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(losses2, 'b-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('F2: Training Loss — Data-Driven RNN', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Skipping — PyTorch not available.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F2.2  Compare RNN hidden representations to real population data
#       using CKA (Centered Kernel Alignment) and RSA
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    from sklearn.decomposition import PCA
    
    pred_model.eval()
    
    # ── Get hidden states and predictions on test set ──
    with torch.no_grad():
        pred_test, hidden_test = pred_model(X_test)
    
    pred_np = pred_test.cpu().numpy().reshape(-1, n_target)  # (n_test_trials, n_cells)
    hidden_np = hidden_test.cpu().numpy().reshape(-1, 256)   # (n_test_trials, n_hidden)
    actual_np = Y_test.cpu().numpy().reshape(-1, n_target)   # (n_test_trials, n_cells)
    
    # ── CKA: compare real population vs RNN hidden states ──
    def linear_CKA(X1, X2):
        """Compute linear CKA between two representations."""
        K1 = X1 @ X1.T
        K2 = X2 @ X2.T
        n = K1.shape[0]
        H = np.eye(n) - np.ones((n, n)) / n
        K1c = H @ K1 @ H
        K2c = H @ K2 @ H
        hsic = np.trace(K1c @ K2c)
        norm = np.sqrt(np.trace(K1c @ K1c) * np.trace(K2c @ K2c))
        return hsic / norm if norm > 0 else 0
    
    # Subsample time points for efficiency
    n_sub = min(500, actual_np.shape[0])
    idx = np.random.choice(actual_np.shape[0], n_sub, replace=False)
    
    cka_actual_hidden = linear_CKA(actual_np[idx], hidden_np[idx])
    cka_actual_pred = linear_CKA(actual_np[idx], pred_np[idx])
    
    print(f"CKA (real activity vs RNN hidden): {cka_actual_hidden:.4f}")
    print(f"CKA (real activity vs RNN predicted): {cka_actual_pred:.4f}")
    
    # ── PCA comparison: real vs RNN hidden ──
    pca_real = PCA(n_components=3).fit_transform(actual_np[idx])
    pca_hidden = PCA(n_components=3).fit_transform(hidden_np[idx])
    
    # Get stimulus labels for coloring
    test_stim = X_test.cpu().numpy().reshape(-1, 5)[idx]
    test_ori_label = np.round(np.rad2deg(np.arctan2(test_stim[:, 1], test_stim[:, 0])) % 360).astype(int)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    ax = axes[0]
    sc0 = ax.scatter(pca_real[:, 0], pca_real[:, 1], c=test_ori_label, cmap='hsv', s=10, alpha=0.5)
    ax.set_title('Real Population PCA', fontweight='bold')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc0, ax=ax, label='Orientation (°)')
    
    ax = axes[1]
    sc1 = ax.scatter(pca_hidden[:, 0], pca_hidden[:, 1], c=test_ori_label, cmap='hsv', s=10, alpha=0.5)
    ax.set_title('RNN Hidden States PCA', fontweight='bold')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc1, ax=ax, label='Orientation (°)')
    
    # Learned recurrent weights analysis
    ax = axes[2]
    rnn_weights = pred_model.rnn.weight_hh_l0.detach().cpu().numpy()
    im = ax.imshow(rnn_weights, aspect='auto', cmap='RdBu_r',
                   vmin=-np.percentile(np.abs(rnn_weights), 95),
                   vmax=np.percentile(np.abs(rnn_weights), 95))
    plt.colorbar(im, ax=ax)
    ax.set_title('F2: Learned Recurrent Weights', fontweight='bold')
    ax.set_xlabel('From unit')
    ax.set_ylabel('To unit')
    
    plt.suptitle(f'F2: Data-Driven RNN Analysis (CKA={cka_actual_hidden:.3f})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # ── Cell-by-cell prediction quality ──
    per_cell_r = []
    for ci in range(n_target):
        r, _ = pearsonr(actual_np[:, ci], pred_np[:, ci])
        per_cell_r.append(r)
    per_cell_r = np.array(per_cell_r)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ax = axes[0]
    ax.hist(per_cell_r, bins=30, color='steelblue', alpha=0.7, edgecolor='k')
    ax.axvline(np.median(per_cell_r), color='red', ls='--', label=f'median={np.median(per_cell_r):.3f}')
    ax.set_xlabel('Pearson r (predicted vs actual)')
    ax.set_ylabel('# Cells')
    ax.set_title('F2: Per-Cell Prediction Quality', fontweight='bold')
    ax.legend()
    
    # By subclass
    ax = axes[1]
    pred_df = pd.DataFrame({'r': per_cell_r, 'subclass': target_subclasses})
    for sc in present_subclasses:
        vals = pred_df.loc[pred_df['subclass'] == sc, 'r'].values
        if len(vals) >= 3:
            ax.scatter([SUBCLASS_SHORT[sc]] * len(vals), vals, alpha=0.5, s=20, c=SUBCLASS_COLORS[sc])
            ax.scatter(SUBCLASS_SHORT[sc], np.mean(vals), s=100, c=SUBCLASS_COLORS[sc],
                      edgecolors='k', zorder=5, marker='D')
    ax.set_xlabel('Subclass')
    ax.set_ylabel('Prediction r')
    ax.set_title('F2: Prediction Quality by Cell Type', fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Skipping — PyTorch not available.")

## F3: Temporal Trajectory RNN (10 Hz ΔF/F)

Using the 10 Hz trial-resolved traces, we train an RNN to predict the **full temporal trajectory** of population ΔF/F within each trial (41 time steps from -1s to +3s), rather than just the trial-averaged scalar. This enables:
- Comparison of temporal dynamics between model and real neurons
- Assessing whether the RNN learns transient/sustained response profiles
- CKA comparison at each time point within the trial

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F3.1  Load 10 Hz data & train RNN on full temporal trajectories
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    def load_zarr_10hz(mouse_id, session='session_1'):
        """Load 10 Hz trial-resolved data from zarr."""
        z = zarr.open(f'{ZARR_DIR}/{mouse_id}_multimodal_data.zarr', 'r')
        gs = z[f'ophys/drifting_gratings/{session}/stim_aligned_dff/GratingStim']
        ti_dict = {k: gs[f'trial_info/{k}'][:] for k in gs['trial_info'].keys()}
        return {
            'dff': gs['dff'][:],
            'unique_ids': z['unique_id'][:].astype(str),
            'running': gs['running'][:],
            'time_rel': gs['time_relative'][:],
            'trial_info': pd.DataFrame(ti_dict),
        }
    
    # Load from the largest mouse
    mouse_f3 = obs['mouse_id'].value_counts().idxmax()
    pk = load_zarr_10hz(mouse_f3)
    
    dff_10hz = pk['dff']              # (n_trials, 41, n_cells)
    run_10hz = pk['running'][:, :, 0] # (n_trials, 41) speed
    time_rel_f3 = pk['time_rel']      # (41,)
    ti_f3 = pk['trial_info']
    uids_f3 = pk['unique_ids']
    
    n_trials_f3, n_tp_f3, n_cells_f3 = dff_10hz.shape
    
    # Non-gray grating trials
    non_gray_f3 = ~ti_f3['gray_screen'].astype(bool)
    grating_idx_f3 = np.where(non_gray_f3.values)[0]
    n_gt_f3 = len(grating_idx_f3)
    
    # Subsample cells for tractability
    np.random.seed(42)
    n_target_f3 = min(80, n_cells_f3)
    target_cells_f3 = np.sort(np.random.choice(n_cells_f3, n_target_f3, replace=False))
    
    # Match subclasses
    obs_mouse_f3 = obs[obs['mouse_id'] == mouse_f3]
    target_sc_f3 = []
    for ci in target_cells_f3:
        uid = uids_f3[ci]
        m = obs_mouse_f3[obs_mouse_f3['unique_id'] == uid]
        target_sc_f3.append(m.iloc[0]['subclass_name'] if len(m) > 0 else 'Unknown')
    target_sc_f3 = np.array(target_sc_f3)
    
    # ── Build input/output for RNN ──
    ori_f3 = ti_f3.loc[grating_idx_f3, 'orientation'].values.astype(float)
    contrast_f3 = ti_f3.loc[grating_idx_f3, 'contrast'].values.astype(float)
    tf_f3 = ti_f3.loc[grating_idx_f3, 'temporal_frequency'].values.astype(float)
    
    X_input_f3 = np.zeros((n_gt_f3, n_tp_f3, 6))
    for tr in range(n_gt_f3):
        ori_rad = np.deg2rad(ori_f3[tr])
        X_input_f3[tr, :, 0] = np.cos(ori_rad)
        X_input_f3[tr, :, 1] = np.sin(ori_rad)
        X_input_f3[tr, :, 2] = contrast_f3[tr] / 0.8
        X_input_f3[tr, :, 3] = np.log2(tf_f3[tr]) / 4
        X_input_f3[tr, :, 4] = np.clip(run_10hz[grating_idx_f3[tr]] / 30, -1, 2)
        X_input_f3[tr, :, 5] = time_rel_f3 / 3.0
    
    Y_output_f3 = dff_10hz[grating_idx_f3][:, :, target_cells_f3]
    
    # Train/test split
    perm_f3 = np.random.permutation(n_gt_f3)
    n_train_f3 = int(0.8 * n_gt_f3)
    train_f3 = perm_f3[:n_train_f3]
    test_f3 = perm_f3[n_train_f3:]
    
    X_train_f3 = torch.FloatTensor(X_input_f3[train_f3]).to(device)
    X_test_f3 = torch.FloatTensor(X_input_f3[test_f3]).to(device)
    Y_train_f3 = torch.FloatTensor(Y_output_f3[train_f3]).to(device)
    Y_test_f3 = torch.FloatTensor(Y_output_f3[test_f3]).to(device)
    
    # Define temporal RNN
    class TemporalRNN(nn.Module):
        """RNN that predicts population ΔF/F at every 100ms within a trial."""
        def __init__(self, n_input=6, n_hidden=256, n_output=80, n_layers=2, dropout=0.1):
            super().__init__()
            self.rnn = nn.GRU(n_input, n_hidden, n_layers, batch_first=True, dropout=dropout)
            self.readout = nn.Sequential(
                nn.Linear(n_hidden, n_hidden // 2),
                nn.ReLU(),
                nn.Linear(n_hidden // 2, n_output),
            )
        def forward(self, x):
            out, _ = self.rnn(x)
            return self.readout(out), out
    
    temporal_model = TemporalRNN(n_input=6, n_hidden=256, n_output=n_target_f3).to(device)
    opt_f3 = optim.Adam(temporal_model.parameters(), lr=1e-3)
    
    # Training
    batch_size_f3 = 128
    n_epochs_f3 = 150
    losses_f3 = []
    
    print(f"Training temporal trajectory RNN: {n_train_f3} trials, 41 timepoints, {n_target_f3} cells")
    for epoch in range(n_epochs_f3):
        temporal_model.train()
        epoch_loss = 0
        idx_shuffled = np.random.permutation(n_train_f3)
        
        for start in range(0, n_train_f3, batch_size_f3):
            batch = idx_shuffled[start:start+batch_size_f3]
            pred, _ = temporal_model(X_train_f3[batch])
            loss = nn.MSELoss()(pred, Y_train_f3[batch])
            
            opt_f3.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(temporal_model.parameters(), 1.0)
            opt_f3.step()
            epoch_loss += loss.item()
        
        losses_f3.append(epoch_loss / max(1, n_train_f3 // batch_size_f3))
        
        if (epoch + 1) % 30 == 0:
            temporal_model.eval()
            with torch.no_grad():
                pred_test_f3, _ = temporal_model(X_test_f3)
                test_loss_f3 = nn.MSELoss()(pred_test_f3, Y_test_f3).item()
            print(f"  Epoch {epoch+1}: train={losses_f3[-1]:.4f}, test={test_loss_f3:.4f}")
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(losses_f3, 'b-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('F3: Training Loss — Temporal Trajectory RNN', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Skipping — PyTorch not available.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F3.2  Compare temporal dynamics: predicted vs real PSTH, time-resolved CKA
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    from sklearn.decomposition import PCA
    
    temporal_model.eval()
    with torch.no_grad():
        pred_f3, hidden_f3 = temporal_model(X_test_f3)
    
    pred_f3_np = pred_f3.cpu().numpy()      # (n_test, 41, n_target)
    hidden_f3_np = hidden_f3.cpu().numpy()  # (n_test, 41, n_hidden)
    actual_f3_np = Y_test_f3.cpu().numpy()  # (n_test, 41, n_target)
    n_test_f3 = actual_f3_np.shape[0]
    
    # ── 1. Compare mean predicted vs real PSTH (averaged across test trials) ──
    mean_pred_psth = np.nanmean(pred_f3_np, axis=0)   # (41, n_target)
    mean_real_psth = np.nanmean(actual_f3_np, axis=0)  # (41, n_target)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Per-subclass mean PSTH: real vs predicted
    present_sc_f3 = [s for s in SUBCLASS_ORDER if s in target_sc_f3]
    for ax, sc in zip(axes[0], present_sc_f3[:3]):
        sc_mask = target_sc_f3 == sc
        if sc_mask.sum() < 2:
            continue
        real_mean = np.nanmean(mean_real_psth[:, sc_mask], axis=1)
        pred_mean = np.nanmean(mean_pred_psth[:, sc_mask], axis=1)
        ax.plot(time_rel_f3, real_mean, color=SUBCLASS_COLORS[sc], linewidth=2, label='Real')
        ax.plot(time_rel_f3, pred_mean, color=SUBCLASS_COLORS[sc], linewidth=2, ls='--', label='Predicted')
        ax.axvline(0, color='k', ls='--', alpha=0.3)
        ax.axvline(2.0, color='k', ls=':', alpha=0.3)
        ax.set_title(f'{SUBCLASS_SHORT[sc]} (n={sc_mask.sum()})',
                     color=SUBCLASS_COLORS[sc], fontweight='bold')
        ax.set_xlabel('Time (s)')
        ax.legend(fontsize=8)
    axes[0, 0].set_ylabel('ΔF/F')
    
    # Handle overflow subclasses
    for ax, sc in zip(axes[1][:len(present_sc_f3[3:])], present_sc_f3[3:]):
        sc_mask = target_sc_f3 == sc
        if sc_mask.sum() < 2:
            continue
        real_mean = np.nanmean(mean_real_psth[:, sc_mask], axis=1)
        pred_mean = np.nanmean(mean_pred_psth[:, sc_mask], axis=1)
        ax.plot(time_rel_f3, real_mean, color=SUBCLASS_COLORS[sc], linewidth=2, label='Real')
        ax.plot(time_rel_f3, pred_mean, color=SUBCLASS_COLORS[sc], linewidth=2, ls='--', label='Predicted')
        ax.axvline(0, color='k', ls='--', alpha=0.3)
        ax.set_title(f'{SUBCLASS_SHORT[sc]} (n={sc_mask.sum()})',
                     color=SUBCLASS_COLORS[sc], fontweight='bold')
        ax.legend(fontsize=8)
    
    # ── 2. Time-resolved CKA ──
    def linear_CKA(X1, X2):
        K1, K2 = X1 @ X1.T, X2 @ X2.T
        n = K1.shape[0]
        H = np.eye(n) - np.ones((n, n)) / n
        K1c, K2c = H @ K1 @ H, H @ K2 @ H
        hsic = np.trace(K1c @ K2c)
        norm = np.sqrt(np.trace(K1c @ K1c) * np.trace(K2c @ K2c))
        return hsic / norm if norm > 0 else 0
    
    cka_per_tp = np.zeros(n_tp_f3)
    n_sub_cka = min(300, n_test_f3)
    cka_idx = np.random.choice(n_test_f3, n_sub_cka, replace=False)
    
    for tp in range(n_tp_f3):
        real_tp = actual_f3_np[cka_idx, tp, :]   # (n_sub, n_cells)
        hidden_tp = hidden_f3_np[cka_idx, tp, :]  # (n_sub, n_hidden)
        cka_per_tp[tp] = linear_CKA(real_tp, hidden_tp)
    
    ax = axes[1, -2] if len(present_sc_f3) <= 3 else axes[1, len(present_sc_f3[3:])]
    ax.plot(time_rel_f3, cka_per_tp, 'b-', linewidth=2, marker='o', markersize=3)
    ax.axvline(0, color='k', ls='--', alpha=0.4)
    ax.axvline(2.0, color='k', ls=':', alpha=0.4)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('CKA')
    ax.set_title('F3: Time-Resolved CKA\n(Real vs RNN Hidden)', fontweight='bold')
    ax.set_ylim(0, max(0.5, cka_per_tp.max() * 1.2))
    
    # ── 3. Per-cell temporal prediction quality ──
    per_cell_temporal_r = np.zeros(n_target_f3)
    for ci in range(n_target_f3):
        real_flat = actual_f3_np[:, :, ci].ravel()
        pred_flat = pred_f3_np[:, :, ci].ravel()
        valid = ~np.isnan(real_flat) & ~np.isnan(pred_flat)
        if valid.sum() > 10:
            per_cell_temporal_r[ci] = np.corrcoef(real_flat[valid], pred_flat[valid])[0, 1]
    
    ax_last = axes.flat[-1]
    for sc in present_sc_f3:
        sc_mask = target_sc_f3 == sc
        vals = per_cell_temporal_r[sc_mask]
        if len(vals) >= 2:
            ax_last.scatter([SUBCLASS_SHORT[sc]] * len(vals), vals, alpha=0.5, s=20,
                           color=SUBCLASS_COLORS[sc])
            ax_last.scatter(SUBCLASS_SHORT[sc], np.mean(vals), s=100, color=SUBCLASS_COLORS[sc],
                           edgecolors='k', zorder=5, marker='D')
    ax_last.axhline(0, color='k', ls='--', alpha=0.4)
    ax_last.set_ylabel('Pearson r (temporal)')
    ax_last.set_title('F3: Per-Cell Temporal Prediction', fontweight='bold')
    ax_last.tick_params(axis='x', rotation=45)
    
    # Hide unused axes
    for ax in axes.flat:
        if not ax.has_data():
            ax.set_visible(False)
    
    plt.suptitle('F3: Temporal Trajectory RNN Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\nMedian per-cell temporal r: {np.median(per_cell_temporal_r):.3f}")
    print(f"Mean time-resolved CKA: {np.mean(cka_per_tp):.4f}")
    print(f"Peak CKA at t={time_rel_f3[np.argmax(cka_per_tp)]:.1f}s: {np.max(cka_per_tp):.4f}")
else:
    print("⚠️ Skipping — PyTorch not available.")